# Field Stem Position Verification

This notebook checks the local→UTM coordinate transform for field trees with Vertex/Maßband heights.

**Inputs (project root):**
- `155_EBEe_mitVertexund Maßband.xlsx` – field trees with local plot coordinates and heights.
- `Parzelleneckpunkte_155.xlsx` – plot corner points (4 per Parzelle) with local `x`/`y` and RTK `UTM_x`/`UTM_y` (EPSG:25832).

**Steps:**
1. Load field trees (33 rows with Vertex and/or Maßband heights) and transform local→UTM coordinates with a per-Parzelle affine fit to the four corner points.
2. Visualize the transformed points on before/after-harvest orthomosaics for visual QA.
3. Export the trees with all Excel attributes as `outputs/tables/field_trees_155.shp` for checking in GIS.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds

# Locate project root (the directory containing 'modules/') by walking up from cwd,
# since Jupyter's cwd is typically the notebook's own directory (notebooks/), not the project root.
def _find_project_root(start='.'):
    d = os.path.abspath(start)
    for _ in range(5):
        if os.path.isdir(os.path.join(d, 'modules')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    raise RuntimeError("Could not locate project root containing 'modules/' directory.")

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'modules'))
print(f"Project root: {PROJECT_ROOT}")

from field_data_excel import load_field_trees_with_heights
from coord_transform import fit_parzelle_transforms, transform_trees_local_to_utm

# Paths (input data root: HALO_DATA_ROOT or data_root.local, see README)
from notebook_common import EXCEL_PATHS, CORNERS_PATH, ORTHO_BEFORE, ORTHO_AFTER
EXCEL_PATH = EXCEL_PATHS[0]   # sheet with the Vertex and tape heights (Parzellen 1-26)

print("Step 1: Load field trees and corner points...")
trees_df = load_field_trees_with_heights(EXCEL_PATH)
print(f"Loaded {len(trees_df)} trees with Vertex/Maßband heights")
print(f"Unique Parzellen: {sorted(trees_df['parzelle'].unique(), key=int)}")
print(f"Trees with both Vertex & Maßband: {(trees_df['height_vertex'].notna() & trees_df['height_massband'].notna()).sum()}")

# Corner points: columns [Parzelle, Nr, x, y, Einmessung, UTM_x, UTM_y, UTM_z], 4 per Parzelle
corners_df = pd.read_excel(CORNERS_PATH)
print(f"\nLoaded {len(corners_df)} corner points for {corners_df['Parzelle'].nunique()} Parzellen")

print("\nStep 2: Transform local → UTM coordinates...")
trees_utm = transform_trees_local_to_utm(trees_df, corners_df)
print(f"Transformation complete.")
print(f"Trees with valid UTM coords: {trees_utm['utm_x'].notna().sum()} of {len(trees_utm)}")

# Corner fit quality for the Parzellen that contain field trees
# (RMSE of the affine fit at the 4 corners; RTK_float corners are less accurate)
transforms = fit_parzelle_transforms(corners_df)
float_corners = corners_df[corners_df['Einmessung'].str.contains('float', na=False)].groupby('Parzelle').size()
print("\nCorner fit per Parzelle:")
for parz in sorted(trees_utm['parzelle'].unique(), key=int):
    print(f"  Parzelle {parz}: RMSE {transforms[parz]['rmse']:.2f} m, RTK_float corners: {float_corners.get(int(parz), 0)}")

# Show sample
print("\nSample of transformed trees:")
print(trees_utm[['tree_id', 'local_x', 'local_y', 'utm_x', 'utm_y']].round(2).head(10).to_string())

In [ ]:
print("Step 3: Visualize stem positions on before/after orthomosaics...")

# Filter to trees with valid UTM coordinates
trees_valid = trees_utm[trees_utm['utm_x'].notna()].copy()

# Corner points of the Parzellen that contain field trees (drawn as plot outlines)
plot_corners = corners_df[corners_df['Parzelle'].isin(trees_valid['parzelle'].astype(int))]

# Area of interest: those Parzellen plus a buffer. The orthos are ~26k x 26k px at 2 cm,
# so only this window is read, downsampled to at most MAX_PX on the long side.
BUFFER_M = 15
MAX_PX = 5000
aoi = (plot_corners['UTM_x'].min() - BUFFER_M, plot_corners['UTM_y'].min() - BUFFER_M,
       plot_corners['UTM_x'].max() + BUFFER_M, plot_corners['UTM_y'].max() + BUFFER_M)

def read_ortho_window(path, bounds, max_px=MAX_PX):
    """Read the RGB window covering `bounds` (left, bottom, right, top), downsampled."""
    with rasterio.open(path) as src:
        window = from_bounds(*bounds, transform=src.transform)
        scale = max(1.0, max(window.width, window.height) / max_px)
        out_shape = (min(src.count, 3), int(window.height / scale), int(window.width / scale))
        rgb = src.read(indexes=list(range(1, out_shape[0] + 1)), window=window,
                       out_shape=out_shape, boundless=True, fill_value=0)
        print(f"  {os.path.basename(path)}: CRS {src.crs}, window {out_shape[2]}x{out_shape[1]} px "
              f"({src.res[0] * scale * 100:.1f} cm/px)")
    return rgb.transpose(1, 2, 0)

print("Loading ortho windows...")
ortho_before = read_ortho_window(ORTHO_BEFORE, aoi)
ortho_after = read_ortho_window(ORTHO_AFTER, aoi)

# Before above after (the block of Parzellen is wide, not tall)
fig, axes = plt.subplots(2, 1, figsize=(18, 20))
panels = [(axes[0], ortho_before, 'Before harvest (2026-02-25)'),
          (axes[1], ortho_after, 'After harvest (2026-03-24)')]

for ax, ortho, title in panels:
    ax.imshow(ortho, extent=[aoi[0], aoi[2], aoi[1], aoi[3]])

    # Plot outlines: corners ordered (0,0) -> (48,0) -> (48,30) -> (0,30)
    for parz, grp in plot_corners.groupby('Parzelle'):
        ring = grp.set_index(['x', 'y']).loc[[(0, 0), (48, 0), (48, 30), (0, 30), (0, 0)]]
        ax.plot(ring['UTM_x'], ring['UTM_y'], color='yellow', linewidth=1)
        ax.annotate(f"P{parz}", (ring['UTM_x'].iloc[:4].mean(), ring['UTM_y'].iloc[:4].mean()),
                    color='yellow', fontsize=12, ha='center', va='center', alpha=0.8)

    ax.scatter(trees_valid['utm_x'], trees_valid['utm_y'], c='red', s=50, marker='x', linewidths=2, label='Field stems')
    for idx, row in trees_valid.iterrows():
        ax.annotate(row['tree_id'], (row['utm_x'], row['utm_y']), fontsize=8, color='red',
                    xytext=(3, 3), textcoords='offset points')
    ax.set_title(title)
    ax.set_xlabel('UTM X')
    ax.set_ylabel('UTM Y')
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.legend()

plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'outputs', 'figures'), exist_ok=True)
fig_path = os.path.join(PROJECT_ROOT, 'outputs', 'figures', 'field_stems_before_after.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Figure saved to {fig_path}")
plt.show()

print("\n✓ Visual QA complete. Review the figure above:")
print("  - Do stem markers fall within/near tree crowns on the 'before' image?")
print("  - Do the marked positions show gaps/changes on the 'after' image for felled trees?")
print("  - If yes, the transform is likely correct. If not, check corner points.")

In [ ]:
print("Step 4: Export field trees as shapefile for checking in GIS...")
import geopandas as gpd

SHP_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'tables', 'field_trees_155.shp')

# Shapefile field names are limited to 10 characters -> short names for every Excel column
SHP_FIELDS = {
    'Versuch': 'versuch',
    'Parzelle': 'parzelle',
    'Nr': 'nr',
    'Artcode': 'artcode',
    'fe_art': 'fe_art',
    'x': 'local_x',
    'y': 'local_y',
    'Jahr': 'jahr',
    'DurchmesserUmfang_mm_2020': 'du_mm_2020',
    'height_m_2020': 'h_m_2020',
    'Kronenansatz': 'kra_2020',
    'removal': 'removal',
    'DurchmesserUmfang_2025': 'du_2025',
    'Höhe_2025': 'h_2025',
    'Kronenanst_25': 'kra_2025',
    'Vertexlänge_cm': 'vertex_cm',
    'Maßbandlänge_cm': 'massbd_cm',
    'Kronenansatz_cm': 'kra_cm',
    'Stumpfhöhe_cm': 'stumpf_cm',
    'Gesamthöhe_m_Vertex': 'h_vertex_m',
    'Gesamtlänge_m_Maßband': 'h_massb_m',
    'removal_2026': 'rem_2026',
    'Bken': 'bken',
    'Bemerkung': 'bemerkung',
    'Unnamed: 24': 'notiz',  # free-text notes in the last, unnamed Excel column
}
INT_FIELDS = ['versuch', 'parzelle', 'nr', 'artcode', 'jahr', 'rem_2026']

# All Excel columns for the trees with Vertex/Maßband heights (same filter as the loader)
raw = pd.read_excel(EXCEL_PATH, sheet_name=0)
raw = raw[raw[['Gesamthöhe_m_Vertex', 'Gesamtlänge_m_Maßband']].notna().any(axis=1)]
unmapped = [c for c in raw.columns if c not in SHP_FIELDS]
if unmapped:
    print(f"Warning: Excel columns without a shapefile field name (not exported): {unmapped}")
attrs = raw[[c for c in SHP_FIELDS if c in raw.columns]].rename(columns=SHP_FIELDS)

# Excel columns with mixed content (numbers and text) -> numeric where possible, otherwise text
for col in attrs.columns:
    if attrs[col].dtype.kind == 'f':
        continue
    values = attrs[col].map(lambda v: v.strip() if isinstance(v, str) else v).replace('', np.nan)
    numeric = pd.to_numeric(values, errors='coerce')
    if numeric.notna().sum() == values.notna().sum():
        attrs[col] = numeric.astype(float)
    else:
        attrs[col] = values.map(lambda v: v if pd.isna(v) else str(v)).astype(object)
for col in INT_FIELDS:
    attrs[col] = attrs[col].astype('Int64')

attrs.insert(0, 'tree_id', attrs['parzelle'].astype(str) + '_' + attrs['nr'].astype(str))
export_df = attrs.merge(trees_valid[['tree_id', 'utm_x', 'utm_y']], on='tree_id', how='inner', validate='one_to_one')
if len(export_df) != len(trees_valid):
    print(f"Warning: {len(trees_valid) - len(export_df)} trees could not be matched to their Excel row.")

trees_gdf = gpd.GeoDataFrame(
    export_df,
    geometry=gpd.points_from_xy(export_df['utm_x'], export_df['utm_y']),
    crs='EPSG:25832',
)
os.makedirs(os.path.dirname(SHP_PATH), exist_ok=True)
trees_gdf.to_file(SHP_PATH, driver='ESRI Shapefile', encoding='utf-8')

print(f"Exported {len(trees_gdf)} trees with {len(trees_gdf.columns) - 1} attributes to {SHP_PATH}")
print("\nShapefile field  <-  Excel column")
for excel_col, shp_col in SHP_FIELDS.items():
    print(f"  {shp_col:<10}  <-  {excel_col}")
print("  tree_id     <-  Parzelle_Nr")
print("  utm_x/utm_y <-  transformed position (EPSG:25832)")